In [3]:
import os
from typing import List
# from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
# model
model = ChatOpenAI(
    model="gpt-4"
)

prompt ="classify the below email :" \
"Subject: URGENT - You Won $1 Million! Click Now!\n" \
"Dear Winner, Congratulations! You have been selected to receive $1 million. Click this link immediately to claim your prize: http://sketchy-site.com\n"

response = model.invoke(prompt)


In [8]:
response.content

'This email can be classified as Spam or Phishing.'

In [12]:
prompt = """You are an email classification system. Classify the following email into one of these categories: Spam, Important, Newsletter, Personal.

Email:
"Subject: URGENT - You Won $1 Million! Click Now!
Dear Winner, Congratulations! You have been selected to receive $1 million. Click this link immediately to claim your prize: http://sketchy-site.com"

Provide:
1. Category
2. Confidence level (0-100%)
3. Reasoning (1 sentence)"""


response = model.invoke(prompt)
output = response.content

In [15]:
type(output)
# clean output
#print line by line
output = output.strip()
for line in output.split("\n"):
    print(line)

1. Category: Spam
2. Confidence level: 100%
3. Reasoning: The email promises a large amount of money and directs the recipient to click on a suspicious link, which are typical characteristics of spam emails.


In [16]:
# bad prompt
prompt_resume = "is this resume good" \
"""Name: Priya Sharma
Experience:
- Python Developer at InfoTech (2020-2024) - 4 years
- Built REST APIs using Flask
- Deployed applications on Heroku (not AWS)
- Used SQLite for small projects
- Basic Git knowledge, mostly solo projects

Skills: Python, Flask, SQLite, HTML/CSS, Git

Education: B.Tech Computer Science (2016-2020)"""

response_resume = model.invoke(prompt_resume)

output_resume = response_resume.content

for line in output_resume.split("\n"):
    print(line)

Your resume is off to a solid start, but there are a few changes you could make to improve its impact:

1. Quantifiable Achievements: In your experience section, try to incorporate quantifiable achievements. For instance, "Built REST APIs using Flask" could be enhanced to "Developed and maintained 10+ REST APIs using Flask, improving system efficiency by 20%".

2. More Details: Elaborate on your projects and experiences. What kind of applications did you deploy on Heroku? What was the impact of using SQLite for small projects?

3. Git Knowledge: Try to avoid using words like 'basic' or 'mostly solo projects'. Employers would want to see how proficient you are with Git. You could say - "Utilized Git for version control in multiple projects".

4. Skills: You may want to categorize your skills into sections such as Programming Languages, Web Development, Databases, etc. This could make it easier for the reader.

5. Contact Information: Don't forget to include your contact information (ema

In [20]:
good_prompt= """You are an expert technical recruiter. Analyze the following resume for a Senior Python Developer position.

Job Requirements:
- 5+ years Python development experience
- Django or Flask framework expertise
- AWS cloud deployment experience
- SQL database skills (PostgreSQL/MySQL)
- Git version control

Resume:
Name: Priya Sharma
Experience:
- Python Developer at InfoTech (2020-2024) - 4 years
- Built REST APIs using Flask
- Deployed applications on Heroku (not AWS)
- Used SQLite for small projects
- Basic Git knowledge, mostly solo projects

Skills: Python, Flask, SQLite, HTML/CSS, Git

Education: B.Tech Computer Science (2016-2020)

Provide:
1. Overall Match Score (0-100)
2. Strengths (2-3 points)
3. Gaps/Concerns (2-3 points)
4. Final Recommendation: Strong Yes / Yes / Maybe / No
5. Brief reasoning (2 sentences)
"""


response_good = model.invoke(good_prompt)
output_good = response_good.content

for line in output_good.split("\n"):
    print(line)

1. Overall Match Score: 70
2. Strengths: 
   - Priya has 4 years of Python development experience and expertise in Flask, which is one of the required frameworks.
   - She has experience in building REST APIs, which might prove beneficial in the development process.
3. Gaps/Concerns:
   - Priya lacks the required 5+ years of Python development experience.
   - She has no experience with AWS cloud deployment or SQL databases like PostgreSQL/MySQL, although she has used SQLite for small projects.
   - Her Git knowledge is basic and mostly used for solo projects, which might be a concern if the role requires extensive and collaborative use of Git.
4. Final Recommendation: Maybe
5. Brief reasoning: While Priya is experienced in Python and Flask which are crucial for the role, her lack of experience in AWS, SQL databases and insufficient Git expertise may pose challenges. If she can quickly learn and adapt to these technologies, she might still be a good fit. However, other candidates with 

In [21]:
# read the data ( excel )
prompt = "You are an expert technical recruiter. Analyze the following resume for a Senior Python Developer position." \
"{jd} and  {resume}"

In [25]:
prompt.format(jd = "something",resume="something")

'You are an expert technical recruiter. Analyze the following resume for a Senior Python Developer position.something and  something'

In [ ]:
emails = [
        "Subject: URGENT - You Won $1 Million! Click Now!\nDear Winner, Congratulations! You have been selected to receive $1 million. Click this link immediately to claim your prize: http://sketchy-site.com",
        "Subject: Important Update Regarding Your Account\nDear User, We noticed unusual activity in your account. Please verify your identity by clicking the link below: http://phishing-site.com",
        "Subject: Exclusive Deal Just for You!\nDear Valued Customer, We have an exclusive offer just for you. Click here to claim your discount: http://spam-site.com",
        "Subject: Your Feedback is Needed!\nDear Customer, We value your opinion. Please take a moment to provide feedback on your recent purchase: http://survey-site.com",
        "Subject : You won the Iphone 18 click here to get it"
    ]

In [66]:
import pandas as pd

emails = pd.Series([
    "Subject: URGENT - You Won $1 Million! Click Now!\nDear Winner, Congratulations! You have been selected to receive $1 million. Click this link immediately to claim your prize: http://sketchy-site.com",
    "Subject: Important Update Regarding Your Account\nDear User, We noticed unusual activity in your account. Please verify your identity by clicking the link below: http://phishing-site.com",
    "Subject: Exclusive Deal Just for You!\nDear Valued Customer, We have an exclusive offer just for you. Click here to claim your discount: http://spam-site.com",
    "Subject: Your Feedback is Needed!\nDear Customer, We value your opinion. Please take a moment to provide feedback on your recent purchase: http://survey-site.com",
    "Subject : You won the Iphone 18 click here to get it"
])

email_df = pd.DataFrame(emails, columns=["email"])
email_df.head(2)

,email
0,Subject: URGENT - You Won $1 Million! Click No...
1,Subject: Important Update Regarding Your Accou...


In [29]:
prompt = """You are an email classification system. Classify the following email into one of these categories: Spam, Important, Newsletter, Personal.

Email: {email} and Provide:
1. Overall Match Score (0-100)
2. Strengths (2-3 points)
3. Gaps/Concerns (2-3 points)
4. Final Recommendation: Strong Yes / Yes / Maybe / No
5. Brief reasoning (2 sentences)"""


# we call to model( invoke)

output = model.invoke(prompt.format(email=email_df.iloc[0]["email"])).content

In [32]:
# find out ? classification, overall match score
output

"Classification: Spam\n\n1. Overall Match Score: 95\n2. Strengths: \n   - The email is unsolicited and contains common spam characteristics such as winning a large sum of money.\n   - The email asks to click on a link to claim the prize, which is a common trick used in spam emails.\n3. Gaps/Concerns: \n   - The email does not contain any personal information that could identify the recipient, a common characteristic of important or personal emails.\n   - The email comes from an unverified source and includes a suspicious link.\n4. Final Recommendation: No\n5. Brief Reasoning: This email is marked as 'urgent', promises a large sum of money, and includes a suspicious link, which are all common characteristics of spam emails. Additionally, it lacks personalization and comes from an unverified source, making it highly unlikely to be a legitimate email."

In [ ]:
class EmailOutput(BaseModel):
    classification: str
    overall_match_score: int
    strengths: List[str]
    gaps_concerns: List[str]
    final_recommendation: str
    brief_reasoning: str

# redefine prompt
prompt = """You are an email classification system. Classify the following email into one of these categories: Spam, Important, Newsletter, Personal.

Email: {email} and Provide:
1. classification
2. overall_match_score
3. gaps_concerns (2-3 points)
4. final_recommendation: Strong Yes / Yes / Maybe / No
5. brief_reasoning (2 sentences)"""


# parser = JsonOutputParser(pydantic_object=EmailOutput)

output = model.invoke(prompt.format(email=email_df.iloc[0]["email"])).content

In [41]:
output

"Classification: Spam\n1. Classification: Spam\n2. Overall Match Score: 95%\n3. Gaps Concerns: The sender of the email is not identifiable, the link provided seems untrustworthy, and the email contains typical language used in scam emails.\n4. Final Recommendation: No\n5. Brief Reasoning: This email seems to be a clear example of a scam or phishing attempt, promising a large sum of money with a sketchy link. It's advisable not to click on the link or to provide any personal information."

In [43]:
# correct way to do with string only
prompt = """You are an email classification system. Classify the following email into one of these categories: Spam, Important, Newsletter, Personal.

Email: {email}

Provide your response STRICTLY in this JSON format (no extra text before or after):
{{
    "classification": "one of: Spam, Important, Newsletter, Personal",
    "overall_match_score": 0-100,
    "strengths": ["point 1", "point 2"],
    "gaps_concerns": ["concern 1", "concern 2", "concern 3"],
    "final_recommendation": "one of: Strong Yes, Yes, Maybe, No",
    "brief_reasoning": "2 sentences explaining decision"
}}

Remember: ONLY return valid JSON, nothing else."""
raw_output = model.invoke(prompt.format(email=email_df.iloc[0]["email"])).content




In [50]:
dict(raw_output)

ValueError: dictionary update sequence element #0 has length 1; 2 is required

In [ ]:
# problem 1 : model output : manual parsing ??
# Problem 2 : output format is not strictly followed
# 

'{\n    "classification": "Spam",\n    "overall_match_score": 90,\n    "strengths": ["Contains typical spam phrases", "Website URL appears untrustworthy"],\n    "gaps_concerns": ["Email lacks personalized content", "Email source is not recognized", "Prize winning without any prior participation"],\n    "final_recommendation": "No",\n    "brief_reasoning": "This email is categorized as spam due to its use of typical spam phrases and an untrustworthy link. Additionally, the lack of personalized content and the fact that the recipient won a prize without participating in anything is suspicious."\n}'

In [54]:
# thats why we need langchain
from langchain_core.prompts import PromptTemplate

parser = JsonOutputParser(pydantic_object=EmailOutput)

prompt = PromptTemplate(
    template= """You are an email classification system. 
    Classify the following email into one of these categories: Spam, Important, Newsletter, Personal.

    Email: {email}
    {format_instructions}

    """,
    input_variables=["email"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = prompt | model | parser
output = chain.invoke({"email": email_df.iloc[0]["email"]})


In [55]:
output

{'classification': 'Spam',
 'overall_match_score': 95,
 'strengths': ['Contains phrases typically found in spam emails',
  'Urgency in the subject line',
  'Unverified links'],
 'gaps_concerns': ['Lack of personalization', 'Unknown sender'],
 'final_recommendation': 'Move to spam folder',
 'brief_reasoning': "The email uses phrases commonly found in spam emails such as 'URGENT' and 'You Won $1 Million'. It also contains an unverified link and lacks personalization, which are common characteristics of spam emails."}

In [ ]:
# lets add the output into email df
# email_df.loc[0,'classification'] = output['classification']
# email_df.loc[0,'overall_match_score'] = output['overall_match_score']
# email_df.loc[0,'final_recommendation'] = output['final_recommendation']

In [67]:
# email_df.head()

In [68]:
# try to all in one go
from tqdm import tqdm 

email_df['classification'] = None
email_df['match_score'] = None
email_df['strengths'] = None
email_df['gaps_concerns'] = None
email_df['recommendation'] = None
email_df['reasoning'] = None
email_df['status'] = None 

In [80]:
# process each email with progress bar
for idx in tqdm(range(len(email_df)),desc="Processing Emails"):
    try:
        email_text = email_df.loc[idx, 'email']

        # call the model
        raw_output = chain.invoke(prompt.format(email=email_text))
        email_df.loc[idx,'classification'] = raw_output['classification']
        email_df.loc[idx,'match_score'] = raw_output['overall_match_score']
        email_df.loc[idx,'strengths'] = ", ".join(raw_output['strengths'])
        email_df.loc[idx,'gaps_concerns'] = ", ".join(raw_output['gaps_concerns'])
        email_df.loc[idx,'recommendation'] = raw_output['final_recommendation']
        email_df.loc[idx,'reasoning'] = raw_output['brief_reasoning']
        email_df.loc[idx,'status'] = "Processed"
    except Exception as e:
        print(f"Error processing email at index {idx}: {e}")
        email_df.loc[idx,'status'] = "Error"


Processing Emails: 100%|██████████| 5/5 [00:20<00:00,  4.17s/it]


In [81]:
email_df

,email,classification,match_score,strengths,gaps_concerns,recommendation,reasoning,status
0,Subject: URGENT - You Won $1 Million! Click No...,Spam,95,"Use of urgent language, Promise of a large sum...","Source of email is unknown, The link provided ...",This email should be marked as spam and the us...,This email contains several characteristics ty...,Processed
1,Subject: Important Update Regarding Your Accou...,Spam,85,"Contains phishing link, Urgent tone","No official company signature, URL does not ma...",Mark as spam and ignore,The email contains a phishing link and does no...,Processed
2,Subject: Exclusive Deal Just for You!\nDear Va...,Spam,95,"Contains offer, Directed to 'Valued Customer'",Links to suspicious site,Mark as Spam,Email contains an offer from an unknown source...,Processed
3,Subject: Your Feedback is Needed!\nDear Custom...,Important,85,The email is directly addressed to the custome...,"The email includes a link, which could potenti...",The email should be marked as important but th...,The email seems to be legitimate and is direct...,Processed
4,Subject : You won the Iphone 18 click here to ...,Spam,95,"Contains common spam phrase, Unsolicited prize...","Sender details not provided, No personalization",Mark as Spam,The email contains common spam triggers and of...,Processed


In [97]:
from langchain.prompts import FewShotPromptTemplate
# Define examples
examples = [
    {
        "email": "URGENT! You've won $1M! Click here now!!!",
        "output": """{"classification": "Spam", "overall_match_score": 95, "final_recommendation": "No"}"""
    },
    {
        "email": "Hi Rahul, Project deadline moved to Friday. Please confirm. - Manager",
        "output": """{"classification": "Important", "overall_match_score": 90, "final_recommendation": "Strong Yes"}"""
    },
    {
        "email": "Weekly AI Newsletter: Top 5 trends in GenAI this week",
        "output": """{"classification": "Newsletter", "overall_match_score": 75, "final_recommendation": "Yes"}"""
    }
]

example_template = """
Email: {email}
Classification: {classification}
"""


example_prompt = PromptTemplate(
    input_variables=["email", "classification"],
    template=example_template
)



In [98]:
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    
    # Prefix: Introduction and context
    prefix="""You are an expert email classification system. 
Classify emails into: Spam, Important, Newsletter, or Personal.

Here are some examples of correct classifications:""",
    
    # Suffix: Instructions for the new email
    suffix="""
Now classify this new email:

Email: {email}

{format_instructions}""",
    
    input_variables=["email"],
    example_separator="\n" + "="*60 + "\n"
)

In [99]:
few_shot_prompt = few_shot_prompt.partial(
    format_instructions=parser.get_format_instructions()
)


In [100]:
chain = few_shot_prompt | model | parser
test_email = "Team meeting moved to Thursday 2 PM. Update your calendar."
chain.invoke({"email": test_email})


KeyError: 'classification'

In [101]:
few_shot_prompt = few_shot_prompt.partial(
    format_instructions=parser.get_format_instructions()
)

ValidationError: 1 validation error for FewShotPromptTemplate
_serialized
  Extra inputs are not permitted [type=extra_forbidden, input_value={'lc': 1, 'type': 'not_im...'FewShotPromptTemplate'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden

In [96]:
example_prompt = PromptTemplate(
    input_variables=["email"],
    template=example_template,
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

few_shot_prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    input_variables=["email"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)


ValidationError: 1 validation error for FewShotPromptTemplate
suffix
  Field required [type=missing, input_value={'example_prompt': Prompt...ief_reasoning"]}\n```'}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing